# 13. Grafana y logs de Vault: ejemplos para el SOC

Partimos del clúster de **`1_Deploy_Vault_minikube.ipynb`**: contexto `workshop`, namespace `vault`,
tres servidores Vault con TLS y un volumen montado en `/vault/audit`. El port-forward y el root token están en `/tmp/vault/config.env`.

Este notebook instala **Grafana + Loki + Grafana Alloy** y deja un dashboard y reglas de alerta listos.
Después genera actividad real contra Vault usando únicamente usuarios, políticas y secretos ficticios de esta demo.

```text
Vault audit file_path=stdout ─┐
Logs de los contenedores ────┼─ API de logs de Kubernetes → Alloy → Loki → Grafana
Pod de prueba stdout ────────┘
Vault audit file_path=/vault/audit/soc-audit.json → copia local en PVC
```

Las celdas son de shell, como el resto del laboratorio. Hace falta `kubectl`, `vault`, `jq`, `curl` y `openssl`.
No se reinstala Vault.

La recogida incluye el namespace `vault`. Alloy lee stdout y stderr por la API de Kubernetes, como `kubectl logs`.
No son los objetos Events ni los logs del host.

Es un laboratorio de un solo nodo de Loki, con almacenamiento local y acceso web por port-forward a localhost.
No tiene HA ni representa una arquitectura de SIEM para producción. Los logs de auditoría conservan HMAC;
el notebook no activa `log_raw` ni imprime contraseñas o tokens.

In [27]:
%env KUBE_CONTEXT=workshop
%env VAULT_K8S_NAMESPACE=vault
%env OBS_NAMESPACE=vault-observability
%env WORKDIR=/tmp/vault/soc
%env VAULT_PORT=18443
%env GRAFANA_PORT=13000
%env LOKI_PORT=13100
%env OWNER=vault-soc-demo

env: KUBE_CONTEXT=workshop
env: VAULT_K8S_NAMESPACE=vault
env: OBS_NAMESPACE=vault-observability
env: WORKDIR=/tmp/vault/soc
env: VAULT_PORT=18443
env: GRAFANA_PORT=13000
env: LOKI_PORT=13100
env: OWNER=vault-soc-demo


## Preparación

Las funciones comunes (`load_env`, consultas a Loki, login de la demo) quedan en `${WORKDIR}/lab.sh`.
`load_env` carga `/tmp/vault/config.env` y, si ya existe, `${WORKDIR}/demo.env`.
La dirección de Vault de esta demo es el port-forward local `https://127.0.0.1:18443`, no el `8443` del otro cuaderno.

In [28]:
%%bash
mkdir -p "${WORKDIR:-/tmp/vault/soc}"
cat > "${WORKDIR:-/tmp/vault/soc}/lab.sh" << 'ENDLAB'
# Funciones del laboratorio. Cada celda hace: source "$WORKDIR/lab.sh" && load_env

load_env() {
  : "${KUBE_CONTEXT:=workshop}"
  : "${VAULT_K8S_NAMESPACE:=vault}"
  : "${OBS_NAMESPACE:=vault-observability}"
  : "${WORKDIR:=/tmp/vault/soc}"
  : "${VAULT_PORT:=18443}"
  : "${GRAFANA_PORT:=13000}"
  : "${LOKI_PORT:=13100}"
  : "${OWNER:=vault-soc-demo}"
  export KUBE_CONTEXT VAULT_K8S_NAMESPACE OBS_NAMESPACE WORKDIR VAULT_PORT GRAFANA_PORT LOKI_PORT OWNER
  mkdir -p "$WORKDIR"
  set -a
  # shellcheck disable=SC1091
  source /tmp/vault/config.env
  if [ -f "${WORKDIR}/demo.env" ]; then
    # shellcheck disable=SC1091
    source "${WORKDIR}/demo.env"
  fi
  set +a
  unset VAULT_NAMESPACE
  export VAULT_ADDR="https://127.0.0.1:${VAULT_PORT}"
  set -euo pipefail
}

kube() {
  kubectl --context "$KUBE_CONTEXT" "$@"
}

check_owner() {
  local kind="$1" name="$2" ns="$3" owner
  if ! owner=$(kubectl --context "$KUBE_CONTEXT" get "$kind" "$name" -n "$ns" -o jsonpath='{.metadata.labels.app\.kubernetes\.io/part-of}' 2>/dev/null); then
    return 0
  fi
  if [ "$owner" != "$OWNER" ]; then
    echo "Ya existe un recurso ajeno a esta demo: ${kind}/${name}" >&2
    exit 1
  fi
}

now_ns() {
  echo "$(date +%s)000000000"
}

loki_count() {
  local query="$1" start="$2" now end
  now=$(date +%s)
  end="${now}000000000"
  if [ -z "$start" ]; then
    start="$((now - 1800))000000000"
  fi
  curl -sfG "http://127.0.0.1:${LOKI_PORT}/loki/api/v1/query_range" \
    --data-urlencode "query=${query}" \
    --data-urlencode "start=${start}" \
    --data-urlencode "end=${end}" \
    --data-urlencode "limit=1000" \
    --data-urlencode "direction=forward" \
    | jq '[.data.result[].values[]] | length'
}

wait_loki() {
  local query="$1" minimum="$2" start="$3" n=0 count=0
  while [ "$n" -lt 60 ]; do
    count=$(loki_count "$query" "$start")
    if [ "$count" -ge "$minimum" ]; then
      echo "$count"
      return 0
    fi
    n=$((n + 1))
    sleep 2
  done
  echo "Solo llegaron ${count}; se esperaban ${minimum}." >&2
  echo "Consulta: ${query}" >&2
  return 1
}

loki_instant() {
  local query="$1"
  curl -sfG "http://127.0.0.1:${LOKI_PORT}/loki/api/v1/query" \
    --data-urlencode "query=${query}" \
    | jq -r '.data.result[0].value[1] // "0"'
}

loki_last_line() {
  local query="$1" start="$2" now end
  now=$(date +%s)
  end="${now}000000000"
  curl -sfG "http://127.0.0.1:${LOKI_PORT}/loki/api/v1/query_range" \
    --data-urlencode "query=${query}" \
    --data-urlencode "start=${start}" \
    --data-urlencode "end=${end}" \
    --data-urlencode "limit=100" \
    --data-urlencode "direction=forward" \
    | jq -r '[.data.result[].values[]] | last | .[1] // empty'
}

save_demo_env() {
  umask 077
  cat > "${WORKDIR}/demo.env" <<ENDENV
DEMO_ID=${DEMO_ID}
LECTOR_PASSWORD=${LECTOR_PASSWORD}
OPERADOR_PASSWORD=${OPERADOR_PASSWORD}
LECTOR_TOKEN=${LECTOR_TOKEN}
OPERADOR_TOKEN=${OPERADOR_TOKEN}
ENDENV
  chmod 600 "${WORKDIR}/demo.env"
}

login_user() {
  local user="$1" pass="$2" out token
  out=$(vault write -format=json "auth/${DEMO_ID}/login/${user}" password="$pass")
  token=$(printf '%s' "$out" | jq -r '.auth.client_token // empty')
  if [ -z "$token" ]; then
    echo "Login de ${user} sin token" >&2
    return 1
  fi
  if [ "$user" = "lector" ]; then
    LECTOR_TOKEN=$token
  else
    OPERADOR_TOKEN=$token
  fi
  save_demo_env
}
ENDLAB

In [29]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
command -v kubectl >/dev/null
command -v vault >/dev/null
command -v jq >/dev/null
command -v curl >/dev/null
command -v openssl >/dev/null
test -f /tmp/vault/vault.ca
kube get namespace "$VAULT_K8S_NAMESPACE" >/dev/null
POD=$(kube get pods -n "$VAULT_K8S_NAMESPACE" -l 'app.kubernetes.io/name=vault,component=server,vault-active=true' -o jsonpath='{.items[0].metadata.name}')
if [ -z "$POD" ]; then
  POD=$(kube get pods -n "$VAULT_K8S_NAMESPACE" -l 'app.kubernetes.io/name=vault,component=server' -o jsonpath='{.items[0].metadata.name}')
fi
test -n "$POD"
echo "$POD" > "${WORKDIR}/vault-pod"
kube get pod "$POD" -n "$VAULT_K8S_NAMESPACE" -o json \
  | jq -e '.spec.containers[] | select(.name=="vault") | .volumeMounts[] | select(.mountPath=="/vault/audit")' >/dev/null
kube get pod "$POD" -n "$VAULT_K8S_NAMESPACE" -o jsonpath='{.status.conditions[?(@.type=="Ready")].status}' | grep -q True
kube get storageclass -o json | jq -e '.items[] | select(.metadata.annotations["storageclass.kubernetes.io/is-default-class"]=="true")' >/dev/null
echo "Contexto: ${KUBE_CONTEXT}; Vault: ${VAULT_K8S_NAMESPACE}/${POD}"
echo "Se desplegarán Grafana, Loki y Alloy en ${OBS_NAMESPACE}"

Contexto: workshop; Vault: vault/vault-0
Se desplegarán Grafana, Loki y Alloy en vault-observability


## Configuración de logs, dashboard y alertas

Alloy usa permisos de lectura de pods y `pods/log` en el namespace `vault`. No monta el disco del nodo.
Reconoce el JSON de auditoría por `type=request/response` y `request.id` dentro del contenedor `vault`.
No hace falta habilitar prefijos de auditoría en Vault 2.x.
El resto de líneas se conserva como `log_kind="container"`. No se indexan tokens, IP, rutas ni IDs de petición como labels.

Las consultas cuentan **respuestas**, para no contar dos veces cada operación. El dashboard conserva también las peticiones
para investigar una respuesta ausente. Un error de autenticación y un error de permisos son casos distintos.
Las alertas son ejemplos con umbrales de laboratorio, no recomendaciones universales para producción.

`jq` solo envuelve esas consultas en el JSON que Grafana aprovisiona: 14 paneles y 8 reglas.

In [30]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
cat > "${WORKDIR}/loki-config.yaml" << 'ENDLOKI'
auth_enabled: false
server:
  http_listen_port: 3100
  grpc_listen_port: 9096
common:
  instance_addr: 127.0.0.1
  path_prefix: /var/loki
  replication_factor: 1
  ring:
    kvstore:
      store: inmemory
  storage:
    filesystem:
      chunks_directory: /var/loki/chunks
      rules_directory: /var/loki/rules
schema_config:
  configs:
    - from: "2024-01-01"
      store: tsdb
      object_store: filesystem
      schema: v13
      index:
        prefix: index_
        period: 24h
limits_config:
  retention_period: 168h
  allow_structured_metadata: true
  max_query_series: 2000
compactor:
  working_directory: /var/loki/compactor
  retention_enabled: true
  delete_request_store: filesystem
analytics:
  reporting_enabled: false
ENDLOKI

In [31]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
cat > "${WORKDIR}/config.alloy" << 'ENDALLOY'
logging {
  level = "info"
}
discovery.kubernetes "pods" {
  role = "pod"
  namespaces {
    names = ["vault"]
  }
}
discovery.relabel "pods" {
  targets = discovery.kubernetes.pods.targets
  rule {
    source_labels = ["__meta_kubernetes_namespace"]
    target_label = "namespace"
  }
  rule {
    source_labels = ["__meta_kubernetes_pod_name"]
    target_label = "pod"
  }
  rule {
    source_labels = ["__meta_kubernetes_pod_container_name"]
    target_label = "container"
  }
  rule {
    target_label = "job"
    replacement = "kubernetes-pods"
  }
}
loki.source.kubernetes "pods" {
  targets = discovery.relabel.pods.output
  forward_to = [loki.process.pods.receiver]
}
loki.process "pods" {
  stage.static_labels {
    values = { log_kind = "container" }
  }
  stage.match {
    selector = "{container=\"vault\"}"
    stage.json {
      expressions = {
        audit_shape = "(type == 'request' || type == 'response') && request.id != `null`",
      }
    }
    stage.labels {
      values = { audit_shape = "" }
    }
    stage.match {
      selector = "{audit_shape=\"true\"}"
      stage.static_labels {
        values = { log_kind = "vault_audit" }
      }
    }
    stage.label_drop {
      values = ["audit_shape"]
    }
  }
  forward_to = [loki.write.local.receiver]
}
loki.write "local" {
  endpoint {
    url = "http://loki:3100/loki/api/v1/push"
  }
}
ENDALLOY

In [32]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
cat > "${WORKDIR}/queries.json" << 'ENDQUERIES'
{
  "audit": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\"",
  "responses": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\"",
  "operational": "{namespace=\"vault\", container=\"vault\", log_kind=\"container\"}",
  "failed_login": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | path=~\"auth/soc-demo-[a-f0-9]+/login/.*\" | error!=\"\"",
  "denied": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | path=~\"soc-demo-[a-f0-9]+/.*\" | error=~\"(?i).*permission denied.*\"",
  "reads": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | path=~\"soc-demo-[a-f0-9]+/data/.*\" | operation=\"read\" | error=\"\"",
  "mutations": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | path=~\"soc-demo-[a-f0-9]+/.*\" | operation=~\"create|update|patch|delete\" | error=\"\"",
  "destructive": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | path=~\"soc-demo-[a-f0-9]+/(delete|destroy|metadata)/.*\" | operation=~\"update|delete\" | error=\"\"",
  "root": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | policies=~\".*\\\"root\\\".*\"",
  "policies": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | path=~\"sys/policies/acl/soc-demo-.*\" | operation=~\"create|update|delete\" | error=\"\"",
  "audit_changes": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | path=~\"sys/audit/.*\" | operation=~\"create|update|delete\" | error=\"\"",
  "heartbeat": "{namespace=\"vault\", container=\"soc-heartbeat\"} |= \"stdout-ok\"",
  "db_audit": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | path=~\"sys/mounts/database|database/(config|roles|creds)/.*|sys/leases/revoke.*\" | error=\"\"",
  "db_ops": "{namespace=\"vault\", container=\"vault\", log_kind=\"container\"} |~ \"successful mount:.*path=database/|revoked lease: lease_id=database/creds/\"",
  "db_prefix_revoke": "{namespace=\"vault\", log_kind=\"vault_audit\"} | json event_type=\"type\", path=\"request.path\", operation=\"request.operation\", error=\"error\", principal=\"auth.display_name\", policies=\"auth.policies\", request_id=\"request.id\", origin=\"request.remote_address\" | __error__=\"\" | event_type=\"response\" | path=~\"sys/leases/revoke-prefix/.+/creds/readonly\" | error=\"\""
}
ENDQUERIES

In [33]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
cat > "${WORKDIR}/build.jq" << 'ENDJQ'
def cq($e): "sum(count_over_time(\($e) [5m])) or vector(0)";
def stat($id; $title; $expr): {
  id: $id, title: $title, type: "stat",
  datasource: {type: "loki", uid: "loki-vault-soc"},
  gridPos: {x: (($id - 1) * 6), y: 0, w: 6, h: 4},
  targets: [{refId: "A", expr: $expr, queryType: "instant"}],
  options: {reduceOptions: {calcs: ["lastNotNull"], fields: "", values: false}},
  fieldConfig: {defaults: {unit: "short"}, overrides: []}
};
def logs($id; $title; $expr; $i): {
  id: $id, title: $title, type: "logs",
  datasource: {type: "loki", uid: "loki-vault-soc"},
  gridPos: {x: (if ($i % 2) == 0 then 0 else 12 end), y: (4 + (($i / 2) | floor) * 8), w: 12, h: 8},
  targets: [{refId: "A", expr: $expr, queryType: "range"}],
  options: {showTime: true, showLabels: false, wrapLogMessage: true, prettifyLogMessage: true, enableLogDetails: true, sortOrder: "Descending"}
};
def rule($uid; $title; $expr; $threshold; $severity; $err): {
  uid: ("vault-soc-" + $uid), title: $title, condition: "C", for: "0s",
  noDataState: "OK", execErrState: $err, isPaused: false,
  labels: {severity: $severity, demo: "vault-soc-demo"},
  annotations: {summary: $title, description: "Regla de laboratorio. Revisar el dashboard Vault · SOC y auditoría."},
  data: [
    {refId: "A", relativeTimeRange: {from: 300, to: 0}, datasourceUid: "loki-vault-soc",
     model: {refId: "A", expr: $expr, queryType: "instant", instant: true,
             datasource: {type: "loki", uid: "loki-vault-soc"}, intervalMs: 1000, maxDataPoints: 43200}},
    {refId: "C", relativeTimeRange: {from: 0, to: 0}, datasourceUid: "__expr__",
     model: {refId: "C", type: "threshold", expression: "A",
             datasource: {type: "__expr__", uid: "__expr__"},
             conditions: [{type: "query", query: {params: ["C"]}, reducer: {type: "last", params: []},
                           evaluator: {type: "gt", params: [$threshold]}, operator: {type: "and"}}]}}
  ]
};
{
  dashboard: {
    uid: "vault-soc", title: "Vault · SOC y auditoría", schemaVersion: 39, version: 1, editable: true,
    tags: ["vault", "soc", "demo"], timezone: "browser",
    time: {from: "now-30m", to: "now"}, refresh: "10s",
    templating: {list: []}, annotations: {list: []},
    panels: [
      stat(1; "Respuestas de Vault · 5m"; cq(.responses)),
      stat(2; "Logins fallidos · 5m"; cq(.failed_login)),
      stat(3; "Accesos denegados · 5m"; cq(.denied)),
      stat(4; "Lecturas de demo · 5m"; cq(.reads)),
      logs(10; "1 · Autenticaciones fallidas"; .failed_login; 0),
      logs(11; "2 · Accesos denegados"; .denied; 1),
      logs(12; "3 · Lecturas y ráfagas"; .reads; 2),
      logs(13; "4 · Cambios y borrados"; .mutations; 3),
      logs(14; "5 · Uso de root"; .root; 4),
      logs(15; "6 · Cambios de políticas"; .policies; 5),
      logs(16; "7 · Cambios de auditoría"; .audit_changes; 6),
      logs(17; "Logs operacionales de Vault"; .operational; 7),
      logs(18; "Auditoría completa · request + response"; .audit; 8),
      logs(19; "Prueba de stdout → Grafana"; .heartbeat; 9),
      logs(20; "8 · Database: engine, role y leases"; .db_audit; 10),
      logs(21; "9 · Database: expiración de leases"; .db_ops; 11)
    ]
  },
  rules: {
    apiVersion: 1,
    groups: [{
      orgId: 1, name: "Vault SOC demo", folder: "Vault SOC", interval: "10s",
      rules: [
        rule("login"; "Fallos de login en la demo"; cq(.failed_login); 4; "warning"; "Error"),
        rule("denied"; "Accesos denegados en la demo"; cq(.denied); 4; "warning"; "Error"),
        rule("burst"; "Ráfaga de lecturas en la demo"; cq(.reads); 19; "warning"; "Error"),
        rule("destroy"; "Borrado o destrucción de secretos de demo"; cq(.destructive); 0; "warning"; "Error"),
        rule("root"; "Uso de la política root en Vault"; cq(.root); 0; "critical"; "Error"),
        rule("policy"; "Cambios de políticas de demo"; cq(.policies); 0; "warning"; "Error"),
        rule("audit"; "Cambios en dispositivos de auditoría"; cq(.audit_changes); 0; "critical"; "Error"),
        rule("prefix"; "Revocación de leases por prefijo creds/readonly"; cq(.db_prefix_revoke); 0; "warning"; "Error"),
        rule("pipeline"; "Sin heartbeat de stdout durante 2 minutos"; ("absent_over_time(" + .heartbeat + " [2m]) or vector(0)"); 0; "critical"; "Alerting")
      ]
    }]
  }
}
ENDJQ

jq -f "${WORKDIR}/build.jq" "${WORKDIR}/queries.json" > "${WORKDIR}/grafana-built.json"
jq '.dashboard' "${WORKDIR}/grafana-built.json" > "${WORKDIR}/vault-soc.json"
jq '.rules' "${WORKDIR}/grafana-built.json" > "${WORKDIR}/rules.json"
echo "Paneles: $(jq '.panels | length' "${WORKDIR}/vault-soc.json")"
echo "Reglas: $(jq '.groups[0].rules | length' "${WORKDIR}/rules.json")"

Paneles: 16
Reglas: 9


In [34]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
cat > "${WORKDIR}/datasources.yaml" << 'ENDDS'
apiVersion: 1
datasources:
  - name: Loki · Kubernetes
    uid: loki-vault-soc
    type: loki
    access: proxy
    url: http://loki:3100
    isDefault: true
    editable: false
    jsonData:
      maxLines: 1000
ENDDS

In [35]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
cat > "${WORKDIR}/providers.yaml" << 'ENDPR'
apiVersion: 1
providers:
  - name: Vault SOC
    orgId: 1
    folder: Vault SOC
    type: file
    disableDeletion: false
    updateIntervalSeconds: 15
    options:
      path: /var/lib/grafana/dashboards
ENDPR

In [36]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
cat > "${WORKDIR}/workloads.yaml" << 'ENDWORK'
apiVersion: v1
kind: Namespace
metadata:
  name: vault-observability
  labels:
    app.kubernetes.io/part-of: vault-soc-demo
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: alloy
  namespace: vault-observability
  labels:
    app.kubernetes.io/part-of: vault-soc-demo
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: loki-data
  namespace: vault-observability
  labels:
    app.kubernetes.io/part-of: vault-soc-demo
spec:
  accessModes: ["ReadWriteOnce"]
  resources:
    requests:
      storage: 5Gi
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: grafana-data
  namespace: vault-observability
  labels:
    app.kubernetes.io/part-of: vault-soc-demo
spec:
  accessModes: ["ReadWriteOnce"]
  resources:
    requests:
      storage: 1Gi
---
apiVersion: v1
kind: Service
metadata:
  name: loki
  namespace: vault-observability
  labels:
    app.kubernetes.io/part-of: vault-soc-demo
spec:
  type: ClusterIP
  selector:
    app: loki
    app.kubernetes.io/part-of: vault-soc-demo
  ports:
    - name: http
      port: 3100
      targetPort: 3100
---
apiVersion: v1
kind: Service
metadata:
  name: grafana
  namespace: vault-observability
  labels:
    app.kubernetes.io/part-of: vault-soc-demo
spec:
  type: ClusterIP
  selector:
    app: grafana
    app.kubernetes.io/part-of: vault-soc-demo
  ports:
    - name: http
      port: 3000
      targetPort: 3000
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: vault-soc-logs
  namespace: vault
  labels:
    app.kubernetes.io/part-of: vault-soc-demo
rules:
  - apiGroups: [""]
    resources: ["pods"]
    verbs: ["get", "list", "watch"]
  - apiGroups: [""]
    resources: ["pods/log"]
    verbs: ["get"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: vault-soc-logs
  namespace: vault
  labels:
    app.kubernetes.io/part-of: vault-soc-demo
subjects:
  - kind: ServiceAccount
    name: alloy
    namespace: vault-observability
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: vault-soc-logs
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: loki
  namespace: vault-observability
  labels:
    app: loki
    app.kubernetes.io/part-of: vault-soc-demo
spec:
  replicas: 1
  strategy:
    type: Recreate
  selector:
    matchLabels:
      app: loki
      app.kubernetes.io/part-of: vault-soc-demo
  template:
    metadata:
      labels:
        app: loki
        app.kubernetes.io/part-of: vault-soc-demo
      annotations:
        soc-demo/config-hash: CONFIG_HASH
    spec:
      automountServiceAccountToken: false
      securityContext:
        runAsNonRoot: true
        runAsUser: 10001
        runAsGroup: 10001
        fsGroup: 10001
        seccompProfile:
          type: RuntimeDefault
      containers:
        - name: loki
          image: grafana/loki:3.7.8
          imagePullPolicy: IfNotPresent
          args: ["-config.file=/etc/loki/config.yaml"]
          ports:
            - name: http
              containerPort: 3100
          resources:
            requests:
              cpu: 100m
              memory: 128Mi
            limits:
              cpu: 500m
              memory: 1Gi
          securityContext:
            allowPrivilegeEscalation: false
            capabilities:
              drop: ["ALL"]
          readinessProbe:
            httpGet:
              path: /ready
              port: 3100
            initialDelaySeconds: 5
            periodSeconds: 5
          livenessProbe:
            httpGet:
              path: /ready
              port: 3100
            initialDelaySeconds: 90
            periodSeconds: 15
          volumeMounts:
            - name: config
              mountPath: /etc/loki
              readOnly: true
            - name: data
              mountPath: /var/loki
      volumes:
        - name: config
          configMap:
            name: loki-config
        - name: data
          persistentVolumeClaim:
            claimName: loki-data
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: alloy
  namespace: vault-observability
  labels:
    app: alloy
    app.kubernetes.io/part-of: vault-soc-demo
spec:
  replicas: 1
  strategy:
    type: Recreate
  selector:
    matchLabels:
      app: alloy
      app.kubernetes.io/part-of: vault-soc-demo
  template:
    metadata:
      labels:
        app: alloy
        app.kubernetes.io/part-of: vault-soc-demo
      annotations:
        soc-demo/config-hash: CONFIG_HASH
    spec:
      serviceAccountName: alloy
      automountServiceAccountToken: true
      securityContext:
        runAsNonRoot: true
        runAsUser: 473
        runAsGroup: 473
        fsGroup: 473
        seccompProfile:
          type: RuntimeDefault
      containers:
        - name: alloy
          image: grafana/alloy:v1.20.1
          imagePullPolicy: IfNotPresent
          args:
            - run
            - --server.http.listen-addr=0.0.0.0:12345
            - --storage.path=/var/lib/alloy
            - /etc/alloy/config.alloy
          ports:
            - name: http
              containerPort: 12345
          resources:
            requests:
              cpu: 100m
              memory: 128Mi
            limits:
              cpu: 500m
              memory: 512Mi
          securityContext:
            allowPrivilegeEscalation: false
            capabilities:
              drop: ["ALL"]
          readinessProbe:
            httpGet:
              path: /-/ready
              port: 12345
            initialDelaySeconds: 5
            periodSeconds: 5
          livenessProbe:
            httpGet:
              path: /-/ready
              port: 12345
            initialDelaySeconds: 90
            periodSeconds: 15
          volumeMounts:
            - name: config
              mountPath: /etc/alloy
              readOnly: true
            - name: data
              mountPath: /var/lib/alloy
      volumes:
        - name: config
          configMap:
            name: alloy-config
        - name: data
          emptyDir: {}
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: grafana
  namespace: vault-observability
  labels:
    app: grafana
    app.kubernetes.io/part-of: vault-soc-demo
spec:
  replicas: 1
  strategy:
    type: Recreate
  selector:
    matchLabels:
      app: grafana
      app.kubernetes.io/part-of: vault-soc-demo
  template:
    metadata:
      labels:
        app: grafana
        app.kubernetes.io/part-of: vault-soc-demo
      annotations:
        soc-demo/config-hash: CONFIG_HASH
    spec:
      automountServiceAccountToken: false
      securityContext:
        runAsNonRoot: true
        runAsUser: 472
        runAsGroup: 472
        fsGroup: 472
        seccompProfile:
          type: RuntimeDefault
      containers:
        - name: grafana
          image: grafana/grafana:13.2.3
          imagePullPolicy: IfNotPresent
          env:
            - name: GF_SECURITY_ADMIN_USER
              value: admin
            - name: GF_SECURITY_ADMIN_PASSWORD
              valueFrom:
                secretKeyRef:
                  name: grafana-admin
                  key: password
            - name: GF_USERS_ALLOW_SIGN_UP
              value: "false"
            - name: GF_AUTH_ANONYMOUS_ENABLED
              value: "false"
            - name: GF_SMTP_ENABLED
              value: "false"
            - name: GF_ANALYTICS_REPORTING_ENABLED
              value: "false"
            - name: GF_ANALYTICS_CHECK_FOR_UPDATES
              value: "false"
          ports:
            - name: http
              containerPort: 3000
          resources:
            requests:
              cpu: 100m
              memory: 128Mi
            limits:
              cpu: 500m
              memory: 512Mi
          securityContext:
            allowPrivilegeEscalation: false
            capabilities:
              drop: ["ALL"]
          readinessProbe:
            httpGet:
              path: /api/health
              port: 3000
            initialDelaySeconds: 5
            periodSeconds: 5
          livenessProbe:
            httpGet:
              path: /api/health
              port: 3000
            initialDelaySeconds: 90
            periodSeconds: 15
          volumeMounts:
            - name: data
              mountPath: /var/lib/grafana
            - name: datasources
              mountPath: /etc/grafana/provisioning/datasources
              readOnly: true
            - name: providers
              mountPath: /etc/grafana/provisioning/dashboards
              readOnly: true
            - name: dashboards
              mountPath: /var/lib/grafana/dashboards
              readOnly: true
            - name: alerting
              mountPath: /etc/grafana/provisioning/alerting
              readOnly: true
      volumes:
        - name: data
          persistentVolumeClaim:
            claimName: grafana-data
        - name: datasources
          configMap:
            name: grafana-datasources
        - name: providers
          configMap:
            name: grafana-providers
        - name: dashboards
          configMap:
            name: grafana-dashboards
        - name: alerting
          configMap:
            name: grafana-alerting
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: soc-heartbeat
  namespace: vault
  labels:
    app: soc-heartbeat
    app.kubernetes.io/part-of: vault-soc-demo
spec:
  replicas: 1
  selector:
    matchLabels:
      app: soc-heartbeat
      app.kubernetes.io/part-of: vault-soc-demo
  template:
    metadata:
      labels:
        app: soc-heartbeat
        app.kubernetes.io/part-of: vault-soc-demo
    spec:
      automountServiceAccountToken: false
      securityContext:
        runAsNonRoot: true
        runAsUser: 10001
        seccompProfile:
          type: RuntimeDefault
      containers:
        - name: soc-heartbeat
          image: busybox:1.37.0
          imagePullPolicy: IfNotPresent
          command: ["sh", "-c", "while true; do echo '{\"demo\":\"vault-soc-heartbeat\",\"message\":\"stdout-ok\"}'; sleep 10; done"]
          resources:
            requests:
              cpu: 5m
              memory: 8Mi
            limits:
              cpu: 50m
              memory: 32Mi
          securityContext:
            allowPrivilegeEscalation: false
            capabilities:
              drop: ["ALL"]
ENDWORK

## Desplegar Grafana, Loki y Alloy

Se crean el namespace `vault-observability`, dos PVC (5 GiB + 1 GiB), los tres componentes,
los permisos de lectura de logs y un pod que escribe un heartbeat a stdout cada 10 segundos.
Una réplica de cada componente; reserva unos 2 GiB adicionales de RAM en Minikube.

La contraseña de `admin` se genera una vez y se guarda en el Secret `grafana-admin` y en
`${WORKDIR}/grafana-admin.password`. No se imprime. Si el Secret ya existe, se reutiliza.
Grafana guarda la contraseña en su PVC: cambiar el Secret no cambia una contraseña ya establecida en esa base de datos.

In [37]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env

if kube get secret grafana-admin -n "$OBS_NAMESPACE" >/dev/null 2>&1; then
  check_owner secret grafana-admin "$OBS_NAMESPACE"
  echo "Se reutiliza el Secret grafana-admin"
else
  openssl rand -base64 18 | tr -d '\n' > "${WORKDIR}/grafana-admin.password"
  chmod 600 "${WORKDIR}/grafana-admin.password"
  kube create secret generic grafana-admin -n "$OBS_NAMESPACE" \
    --from-file=password="${WORKDIR}/grafana-admin.password"
  kube label secret grafana-admin -n "$OBS_NAMESPACE" "app.kubernetes.io/part-of=${OWNER}"
  echo "Contraseña nueva en ${WORKDIR}/grafana-admin.password"
fi

apply_cm() {
  local name="$1" key="$2" file="$3"
  check_owner configmap "$name" "$OBS_NAMESPACE"
  kube create configmap "$name" -n "$OBS_NAMESPACE" \
    --from-file="${key}=${file}" \
    --dry-run=client -o yaml | kube apply -f -
  kube label configmap "$name" -n "$OBS_NAMESPACE" "app.kubernetes.io/part-of=${OWNER}" --overwrite
}

apply_cm loki-config config.yaml "${WORKDIR}/loki-config.yaml"
apply_cm alloy-config config.alloy "${WORKDIR}/config.alloy"
apply_cm grafana-datasources datasources.yaml "${WORKDIR}/datasources.yaml"
apply_cm grafana-providers providers.yaml "${WORKDIR}/providers.yaml"
apply_cm grafana-dashboards vault-soc.json "${WORKDIR}/vault-soc.json"
apply_cm grafana-alerting rules.json "${WORKDIR}/rules.json"

HASH=$(cat \
  "${WORKDIR}/loki-config.yaml" \
  "${WORKDIR}/config.alloy" \
  "${WORKDIR}/datasources.yaml" \
  "${WORKDIR}/providers.yaml" \
  "${WORKDIR}/vault-soc.json" \
  "${WORKDIR}/rules.json" | shasum | awk '{print $1}')
sed "s/CONFIG_HASH/${HASH}/" "${WORKDIR}/workloads.yaml" > "${WORKDIR}/workloads-hashed.yaml"

check_owner deployment loki "$OBS_NAMESPACE"
check_owner deployment alloy "$OBS_NAMESPACE"
check_owner deployment grafana "$OBS_NAMESPACE"
check_owner deployment soc-heartbeat "$VAULT_K8S_NAMESPACE"
kube apply -f "${WORKDIR}/workloads-hashed.yaml"

for name in loki alloy grafana; do
  echo "Esperando a ${name}"
  kube rollout status "deployment/${name}" -n "$OBS_NAMESPACE" --timeout=300s
done
kube rollout status deployment/soc-heartbeat -n "$VAULT_K8S_NAMESPACE" --timeout=120s
echo "Grafana, Loki, Alloy y heartbeat desplegados."

Se reutiliza el Secret grafana-admin
configmap/loki-config unchanged
configmap/loki-config not labeled
configmap/alloy-config unchanged
configmap/alloy-config not labeled
configmap/grafana-datasources unchanged
configmap/grafana-datasources not labeled
configmap/grafana-providers unchanged
configmap/grafana-providers not labeled
configmap/grafana-dashboards unchanged
configmap/grafana-dashboards not labeled
configmap/grafana-alerting configured
configmap/grafana-alerting not labeled
namespace/vault-observability unchanged
serviceaccount/alloy unchanged
persistentvolumeclaim/loki-data unchanged
persistentvolumeclaim/grafana-data unchanged
service/loki unchanged
service/grafana unchanged
role.rbac.authorization.k8s.io/vault-soc-logs unchanged
rolebinding.rbac.authorization.k8s.io/vault-soc-logs unchanged
deployment.apps/loki configured
deployment.apps/alloy configured
deployment.apps/grafana configured
deployment.apps/soc-heartbeat unchanged
Esperando a loki
Waiting for deployment "loki"

## Acceso local

Tres port-forward en segundo plano: Grafana `13000`, Loki `13100` y Vault `18443`.
Si el puerto ya escucha y el servicio responde, se reutiliza. No se mata el proceso que lo ocupa.
Los PID nuevos quedan en `${WORKDIR}/pf-*.pid`.


In [38]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
unset VAULT_NAMESPACE

port_open() {
  (echo >/dev/tcp/127.0.0.1/"$1") >/dev/null 2>&1
}

vault_answers() {
  local code
  set +e
  VAULT_ADDR="https://127.0.0.1:$1" vault status >/dev/null 2>&1
  code=$?
  set -e
  # 0 = unsealed, 2 = sealed. En ambos casos el puerto es Vault.
  [ "$code" -eq 0 ] || [ "$code" -eq 2 ]
}

start_pf() {
  local name="$1" ns="$2" resource="$3" local_port="$4" remote_port="$5" health="${6:-}"
  local pidfile="${WORKDIR}/pf-${name}.pid" old
  if [ -f "$pidfile" ]; then
    old=$(cat "$pidfile")
    if kill -0 "$old" 2>/dev/null && { [ -z "$health" ] || [ "$health" = "vault" ] && vault_answers "$local_port" || { [ "$health" != "vault" ] && curl -sf -o /dev/null --max-time 2 "$health"; }; }; then
      echo "port-forward ${name} ya está en marcha (${old})"
      return 0
    fi
    kill "$old" 2>/dev/null || true
  fi
  if port_open "$local_port"; then
    if [ "$health" = "vault" ] && vault_answers "$local_port"; then
      echo "Vault ya escucha en ${local_port}; se reutiliza."
      return 0
    fi
    if [ -n "$health" ] && [ "$health" != "vault" ] && curl -sf -o /dev/null --max-time 2 "$health"; then
      echo "Puerto ${local_port} ya responde; se reutiliza para ${name}."
      return 0
    fi
    echo "Puerto ${local_port} está ocupado y ${name} no responde en él." >&2
    exit 1
  fi
  kube -n "$ns" port-forward --address=127.0.0.1 "$resource" "${local_port}:${remote_port}" \
    > "${WORKDIR}/pf-${name}.log" 2>&1 &
  echo $! > "$pidfile"
  disown "$!" 2>/dev/null || true
  local i
  i=0
  while [ "$i" -lt 40 ]; do
    if ! kill -0 "$(cat "$pidfile")" 2>/dev/null; then
      echo "Falló el port-forward ${name}:" >&2
      tail -20 "${WORKDIR}/pf-${name}.log" >&2 || true
      exit 1
    fi
    if [ "$health" = "vault" ] && vault_answers "$local_port"; then
      echo "port-forward ${name} listo en ${local_port}"
      return 0
    fi
    if [ -n "$health" ] && [ "$health" != "vault" ] && curl -sf -o /dev/null --max-time 2 "$health"; then
      echo "port-forward ${name} listo en ${local_port}"
      return 0
    fi
    if [ -z "$health" ] && port_open "$local_port"; then
      echo "port-forward ${name} listo en ${local_port}"
      return 0
    fi
    i=$((i + 1))
    sleep 0.5
  done
  echo "No arrancó el port-forward ${name}:" >&2
  tail -20 "${WORKDIR}/pf-${name}.log" >&2 || true
  exit 1
}

start_pf grafana "$OBS_NAMESPACE" service/grafana "$GRAFANA_PORT" 3000 "http://127.0.0.1:${GRAFANA_PORT}/api/health"
start_pf loki "$OBS_NAMESPACE" service/loki "$LOKI_PORT" 3100 "http://127.0.0.1:${LOKI_PORT}/ready"
start_pf vault "$VAULT_K8S_NAMESPACE" "pod/$(cat "${WORKDIR}/vault-pod")" "$VAULT_PORT" 8200 vault

if ! vault token lookup >/dev/null 2>"${WORKDIR}/vault-lookup.err"; then
  echo "Vault no acepta el token en https://127.0.0.1:${VAULT_PORT}:" >&2
  cat "${WORKDIR}/vault-lookup.err" >&2
  exit 1
fi
rm -f "${WORKDIR}/vault-lookup.err"
echo "Grafana: http://127.0.0.1:${GRAFANA_PORT}/d/vault-soc  · usuario admin"
echo "TLS de Vault verificado con la CA de /tmp/vault/vault.ca"

port-forward grafana listo en 13000
port-forward loki listo en 13100
port-forward vault ya está en marcha (14211)
Grafana: http://127.0.0.1:13000/d/vault-soc  · usuario admin
TLS de Vault verificado con la CA de /tmp/vault/vault.ca


In [39]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
count=$(wait_loki "$(jq -r .heartbeat "${WORKDIR}/queries.json")" 1 "")
echo "Lineas de heartbeat en Loki: ${count}"
GPASS=$(kube get secret grafana-admin -n "$OBS_NAMESPACE" -o jsonpath='{.data.password}' | base64 -d)
code=$(curl -s -o /dev/null -w '%{http_code}' -u "admin:${GPASS}" "http://127.0.0.1:${GRAFANA_PORT}/api/dashboards/uid/vault-soc")
test "$code" = "200"
curl -sf -u "admin:${GPASS}" "http://127.0.0.1:${GRAFANA_PORT}/api/datasources/uid/loki-vault-soc/health" | jq -e '.status=="OK"' >/dev/null
rules=$(curl -sf -u "admin:${GPASS}" "http://127.0.0.1:${GRAFANA_PORT}/api/v1/provisioning/alert-rules" | jq '[.[] | select(.uid|startswith("vault-soc-"))] | length')
test "$rules" = "9"
unset GPASS
echo "Verificado: stdout del heartbeat en Loki; dashboard, datasource y nueve reglas en Grafana."

Lineas de heartbeat en Loki: 180
Verificado: stdout del heartbeat en Loki; dashboard, datasource y nueve reglas en Grafana.


## Habilitar la auditoría de Vault en stdout

Primero una copia en `/vault/audit/soc-audit.json` y después la salida `stdout` en JSON, sin prefijo.
Se conservan los dispositivos existentes. Si el nombre ya está configurado con otras opciones, la celda se detiene.
Los dispositivos de auditoría son del clúster: no hay que habilitarlos en cada pod.

Hace falta un token con permisos de administración. El caso de `root` usa el token de `config.env`.

In [40]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env

nstdout=$(vault audit list -format=json | jq '[to_entries[] | select(.value.options.file_path=="stdout")] | length')
if [ "$nstdout" -gt 1 ]; then
  echo "Hay varios dispositivos stdout; revisa los duplicados antes de continuar." >&2
  exit 1
fi

enable_audit() {
  local name="$1" path="$2" fp prefix raw
  if vault audit list -format=json | jq -e --arg p "${name}/" 'has($p)' >/dev/null; then
    fp=$(vault audit list -format=json | jq -r --arg p "${name}/" '.[$p].options.file_path')
    prefix=$(vault audit list -format=json | jq -r --arg p "${name}/" '.[$p].options.prefix // empty')
    raw=$(vault audit list -format=json | jq -r --arg p "${name}/" '.[$p].options.log_raw // "false"')
    if [ "$fp" != "$path" ] || [ -n "$prefix" ] || [ "$raw" != "false" ]; then
      echo "El dispositivo ${name} existe con otra configuración; no se modifica." >&2
      exit 1
    fi
    echo "Se reutiliza ${name}"
  else
    vault write "sys/audit/${name}" - <<END
{"type":"file","description":"vault-soc-demo","options":{"file_path":"${path}","log_raw":"false"}}
END
  fi
}

enable_audit soc-file /vault/audit/soc-audit.json
enable_audit soc-stdout stdout
vault token lookup >/dev/null
count=$(wait_loki "$(jq -r .audit "${WORKDIR}/queries.json")" 1 "")
echo "Eventos de auditoría en Loki: ${count}"
kube logs -n "$VAULT_K8S_NAMESPACE" "$(cat "${WORKDIR}/vault-pod")" -c vault --since=2m --tail=2000 \
  > "${WORKDIR}/vault-stdout.txt"
matches=$(grep -E '"type":"(request|response)"' "${WORKDIR}/vault-stdout.txt" | grep -c '"request":' || true)
rm -f "${WORKDIR}/vault-stdout.txt"
if [ "${matches:-0}" -lt 1 ]; then
  echo "No aparece JSON de auditoría en la salida del contenedor." >&2
  exit 1
fi
echo "Auditoría verificada en la salida del contenedor y en Loki. HMAC conservado."

Se reutiliza soc-file
Se reutiliza soc-stdout
Eventos de auditoría en Loki: 439
Auditoría verificada en la salida del contenedor y en Loki. HMAC conservado.


## Recursos de prueba

Se crea un motor KV v2 y un método userpass llamados `soc-demo-<id>`, dos usuarios y dos políticas.
El lector solo puede leer `allowed/*`. El operador gestiona los secretos de esta demo.
Las contraseñas y los tokens se guardan en `${WORKDIR}/demo.env` (permisos 600) y no se imprimen.

Solo en este userpass se fija el bloqueo en diez fallos, para poder generar seis errores y después un login correcto.
Los demás métodos de autenticación no se tocan.

In [41]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env

if [ ! -f "${WORKDIR}/demo.env" ]; then
  DEMO_ID="soc-demo-$(openssl rand -hex 3)"
  LECTOR_PASSWORD=$(openssl rand -base64 24 | tr -d '\n')
  OPERADOR_PASSWORD=$(openssl rand -base64 24 | tr -d '\n')
  LECTOR_TOKEN=""
  OPERADOR_TOKEN=""
  save_demo_env
  load_env
fi

if ! vault secrets list -format=json | jq -e --arg p "${DEMO_ID}/" 'has($p)' >/dev/null; then
  vault secrets enable -path="$DEMO_ID" -description="$OWNER" kv-v2
fi
if ! vault auth list -format=json | jq -e --arg p "${DEMO_ID}/" 'has($p)' >/dev/null; then
  vault auth enable -path="$DEMO_ID" -description="$OWNER" userpass
  vault write "sys/auth/${DEMO_ID}/tune" - <<END
{"user_lockout_config":{"lockout_threshold":"10","lockout_duration":"1m","lockout_counter_reset":"1m"}}
END
fi

vault policy write "${DEMO_ID}-reader" - <<END
path "${DEMO_ID}/data/allowed/*" {
  capabilities = ["read"]
}
END
vault policy write "${DEMO_ID}-operator" - <<END
path "${DEMO_ID}/*" {
  capabilities = ["create", "read", "update", "patch", "delete", "list"]
}
END

if ! vault read "auth/${DEMO_ID}/users/lector" >/dev/null 2>&1; then
  vault write "auth/${DEMO_ID}/users/lector" \
    password="$LECTOR_PASSWORD" \
    token_policies="${DEMO_ID}-reader" \
    token_ttl=8h token_max_ttl=8h >/dev/null
fi
if ! vault read "auth/${DEMO_ID}/users/operador" >/dev/null 2>&1; then
  vault write "auth/${DEMO_ID}/users/operador" \
    password="$OPERADOR_PASSWORD" \
    token_policies="${DEMO_ID}-operator" \
    token_ttl=8h token_max_ttl=8h >/dev/null
fi

if ! VAULT_TOKEN="${OPERADOR_TOKEN:-}" vault token lookup >/dev/null 2>&1; then
  login_user operador "$OPERADOR_PASSWORD"
fi
if ! VAULT_TOKEN="${LECTOR_TOKEN:-}" vault token lookup >/dev/null 2>&1; then
  login_user lector "$LECTOR_PASSWORD"
fi

for i in $(seq -w 0 29); do
  VAULT_TOKEN="$OPERADOR_TOKEN" vault kv put "${DEMO_ID}/allowed/secret-${i}" value=dato-ficticio-soc demo="$DEMO_ID" >/dev/null
done
VAULT_TOKEN="$OPERADOR_TOKEN" vault kv put "${DEMO_ID}/restricted/finanzas" value=ficticio >/dev/null
echo "Recursos de prueba preparados: ${DEMO_ID}"

Success! Uploaded policy: soc-demo-f642a5-reader
Success! Uploaded policy: soc-demo-f642a5-operator
Recursos de prueba preparados: soc-demo-f642a5


## 1. Fallos de login seguidos de un login correcto

Seis contraseñas incorrectas y después un acceso válido. La alerta salta al superar cuatro fallos en cinco minutos.
La regla cuenta fallos; no correlaciona por sí sola la secuencia fallo y luego éxito.

In [42]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
START=$(now_ns)
i=1
while [ "$i" -le 6 ]; do
  vault write "auth/${DEMO_ID}/login/lector" password="incorrecta-$(openssl rand -hex 8)" >/dev/null 2>&1 || true
  i=$((i + 1))
done
login_user lector "$LECTOR_PASSWORD"
base=$(jq -r .failed_login "${WORKDIR}/queries.json")
count=$(wait_loki "${base} | path=\"auth/${DEMO_ID}/login/lector\"" 6 "$START")
echo "Fallos de login recibidos en Loki: ${count}. El acceso posterior fue correcto."

Fallos de login recibidos en Loki: 6. El acceso posterior fue correcto.


## 2. Accesos denegados

El lector intenta leer `restricted/finanzas`. El token es válido y la política no permite esa ruta.
Se esperan seis respuestas HTTP 403 y seis eventos `permission denied`.

In [43]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
START=$(now_ns)
i=1
while [ "$i" -le 6 ]; do
  if VAULT_TOKEN="$LECTOR_TOKEN" vault kv get "${DEMO_ID}/restricted/finanzas" >/dev/null 2>&1; then
    echo "La lectura debía ser denegada" >&2
    exit 1
  fi
  i=$((i + 1))
done
base=$(jq -r .denied "${WORKDIR}/queries.json")
count=$(wait_loki "${base} | path=\"${DEMO_ID}/data/restricted/finanzas\"" 6 "$START")
echo "Accesos denegados recibidos en Loki: ${count}"

Accesos denegados recibidos en Loki: 6


## 3. Una lectura normal y una ráfaga

Una lectura y después treinta lecturas autorizadas. La alerta mira el **volumen**, no demuestra exfiltración.
La IP es la del port-forward del laboratorio.

In [44]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
START=$(now_ns)
VAULT_TOKEN="$LECTOR_TOKEN" vault kv get "${DEMO_ID}/allowed/secret-00" >/dev/null
i=0
while [ "$i" -le 29 ]; do
  n=$(printf '%02d' "$i")
  VAULT_TOKEN="$LECTOR_TOKEN" vault kv get "${DEMO_ID}/allowed/secret-${n}" >/dev/null
  i=$((i + 1))
done
base=$(jq -r .reads "${WORKDIR}/queries.json")
count=$(wait_loki "${base} | path=~\"${DEMO_ID}/data/allowed/.*\"" 31 "$START")
echo "Lecturas correctas recibidas en Loki: ${count}"

Lecturas correctas recibidas en Loki: 31


## 4. Modificación, borrado recuperable y destrucción definitiva

Sobre una clave ficticia nueva: dos versiones, borrado y recuperación de la segunda, destrucción de la primera
y borrado de los metadatos. En KV v2 hay que mirar **ruta y operación**:
`destroy/` usa HTTP PUT y aparece como `update`.

In [45]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
START=$(now_ns)
key="allowed/ciclo-$(openssl rand -hex 4)"
VAULT_TOKEN="$OPERADOR_TOKEN" vault kv put "${DEMO_ID}/${key}" value=ficticio-v1 >/dev/null
VAULT_TOKEN="$OPERADOR_TOKEN" vault kv put "${DEMO_ID}/${key}" value=ficticio-v2 >/dev/null
VAULT_TOKEN="$OPERADOR_TOKEN" vault write "${DEMO_ID}/delete/${key}" - <<END
{"versions":[2]}
END
VAULT_TOKEN="$OPERADOR_TOKEN" vault write "${DEMO_ID}/undelete/${key}" - <<END
{"versions":[2]}
END
VAULT_TOKEN="$OPERADOR_TOKEN" vault write "${DEMO_ID}/destroy/${key}" - <<END
{"versions":[1]}
END
VAULT_TOKEN="$OPERADOR_TOKEN" vault delete "${DEMO_ID}/metadata/${key}" >/dev/null
base=$(jq -r .mutations "${WORKDIR}/queries.json")
count=$(wait_loki "${base} | path=~\"${DEMO_ID}/.*/${key}\"" 6 "$START")
echo "Operaciones del ciclo KV v2 recibidas en Loki: ${count}"

Success! Data written to: soc-demo-f642a5/delete/allowed/ciclo-9373e1c8
Success! Data written to: soc-demo-f642a5/undelete/allowed/ciclo-9373e1c8
Success! Data written to: soc-demo-f642a5/destroy/allowed/ciclo-9373e1c8
Operaciones del ciclo KV v2 recibidas en Loki: 6


## 5. Uso de root y cambio de permisos

La lectura con el token de administración queda como actividad privilegiada si ese token tiene la política `root`.
Después se amplía solo la política del lector para una ruta antes denegada, se comprueba la lectura y se restaura.
El texto de la política no sale en claro en el audit log: queda el HMAC.

In [46]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
START=$(now_ns)
if vault token lookup -format=json | jq -e '.data.policies | index("root")' >/dev/null; then
  vault kv get "${DEMO_ID}/allowed/secret-00" >/dev/null
  base=$(jq -r .root "${WORKDIR}/queries.json")
  count=$(wait_loki "${base} | path=\"${DEMO_ID}/data/allowed/secret-00\"" 1 "$START")
  echo "Uso de root detectado. Eventos: ${count}"
else
  echo "omitido: el token no tiene política root"
fi

START=$(now_ns)
vault policy read "${DEMO_ID}-reader" > "${WORKDIR}/reader.hcl"
cp "${WORKDIR}/reader.hcl" "${WORKDIR}/reader-extra.hcl"
cat >> "${WORKDIR}/reader-extra.hcl" <<END

path "${DEMO_ID}/data/restricted/finanzas" {
  capabilities = ["read"]
}
END
restore_policy() {
  vault policy write "${DEMO_ID}-reader" "${WORKDIR}/reader.hcl" >/dev/null
}
trap restore_policy EXIT
vault policy write "${DEMO_ID}-reader" "${WORKDIR}/reader-extra.hcl" >/dev/null
VAULT_TOKEN="$LECTOR_TOKEN" vault kv get "${DEMO_ID}/restricted/finanzas" >/dev/null
restore_policy
trap - EXIT
if VAULT_TOKEN="$LECTOR_TOKEN" vault kv get "${DEMO_ID}/restricted/finanzas" >/dev/null 2>&1; then
  echo "El acceso debía volver a estar denegado" >&2
  exit 1
fi
base=$(jq -r .policies "${WORKDIR}/queries.json")
count=$(wait_loki "${base} | path=\"sys/policies/acl/${DEMO_ID}-reader\"" 2 "$START")
echo "Ampliación y restauración de permisos verificadas. Eventos: ${count}. El acceso vuelve a estar denegado."

Uso de root detectado. Eventos: 1
Ampliación y restauración de permisos verificadas. Eventos: 2. El acceso vuelve a estar denegado.


## 6. Cambio en la configuración de auditoría

Se habilita y se retira un tercer dispositivo de fichero. Los dos destinos principales siguen activos.
No se bloquea el disco ni se apaga la auditoría. El fichero temporal queda en el volumen.

In [47]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
START=$(now_ns)
probe="${DEMO_ID}-audit-probe"
if vault audit list -format=json | jq -e --arg p "${probe}/" 'has($p)' >/dev/null; then
  echo "Ya existe el dispositivo temporal." >&2
  exit 1
fi
vault write "sys/audit/${probe}" - <<END
{"type":"file","description":"vault-soc-demo","options":{"file_path":"/vault/audit/${probe}.json","log_raw":"false"}}
END
vault token lookup >/dev/null
vault audit disable "$probe"
base=$(jq -r .audit_changes "${WORKDIR}/queries.json")
count=$(wait_loki "${base} | path=\"sys/audit/${probe}\"" 2 "$START")
echo "Alta y baja del dispositivo temporal recibidas en Loki: ${count}"

Success! Data written to: sys/audit/soc-demo-f642a5-audit-probe
Success! Disabled audit device (if it was enabled) at: soc-demo-f642a5-audit-probe/
Alta y baja del dispositivo temporal recibidas en Loki: 2


## 7. Secretos dinámicos de base de datos

Usa el motor `database` y el role `readonly` creados en `2_Database_Secret_Engine.ipynb` (PostgreSQL, TTL por defecto 1 minuto). No se vuelve a crear la base de datos.

En la **auditoría** (HMAC, sin contraseña en claro) quedan la alta del engine (`sys/mounts/database`), la conexión (`database/config/postgresql`), la configuración del role (`database/roles/readonly`), la emisión de credenciales (`database/creds/readonly`, con `lease_id`) y la revocación manual (`sys/leases/revoke/...`).

En el **log operacional** queda lo que no pide un cliente: la línea `successful mount` al arrancar el motor, y `expiration: revoked lease` cuando el lease caduca solo. Una revocación manual también escribe esa línea, pero además tiene la petición en la auditoría.

In [48]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env

RESP=$(jq -r .responses "${WORKDIR}/queries.json")
OPS=$(jq -r .operational "${WORKDIR}/queries.json")

echo "Motor y role ya existentes (auditoría):"
printf '%s\n' \
  "sys/mounts/database|update|alta del engine" \
  "database/config/postgresql|create|configuración de la conexión" \
  "database/roles/readonly|create|configuración del role" \
  | while IFS='|' read -r path operation label; do
      q="${RESP} | path=\"${path}\" | operation=\"${operation}\""
      echo "  ${label}: $(loki_count "$q" "")"
    done

echo "Montaje del engine en el log operacional:"
echo "  $(loki_count "${OPS} |= \"successful mount:\" |= \"path=database/\"" "")"

echo ""
echo "Lease revocado a mano:"
START=$(now_ns)
out=$(vault read -format=json database/creds/readonly)
lease=$(printf '%s' "$out" | jq -r .lease_id)
dur=$(printf '%s' "$out" | jq -r .lease_duration)
echo "  creado ${lease} (ttl ${dur}s)"
vault lease revoke "$lease" >/dev/null
echo "  auditoría sys/leases/revoke: $(wait_loki "${RESP} | path=\"sys/leases/revoke/${lease}\"" 1 "$START")"
echo "  log operacional: $(wait_loki "${OPS} |= \"revoked lease:\" |= \"${lease}\"" 1 "$START")"

echo ""
echo "Lease que se deja caducar:"
START=$(now_ns)
out=$(vault read -format=json database/creds/readonly)
lease=$(printf '%s' "$out" | jq -r .lease_id)
dur=$(printf '%s' "$out" | jq -r .lease_duration)
echo "  creado ${lease} (ttl ${dur}s). Esperando $((dur + 20))s."
wait_loki "${RESP} | path=\"database/creds/readonly\" | operation=\"read\"" 1 "$START" >/dev/null
sleep $((dur + 20))
echo "  log operacional al expirar: $(wait_loki "${OPS} |= \"revoked lease:\" |= \"${lease}\"" 1 "$START")"
echo "  revocaciones manuales de este lease en auditoría: $(loki_count "${RESP} | path=\"sys/leases/revoke/${lease}\"" "$START")"

Motor y role ya existentes (auditoría):
  alta del engine: 1
  configuración de la conexión: 1
  configuración del role: 1
Montaje del engine en el log operacional:
  3

Lease revocado a mano:
  creado database/creds/readonly/gf6ABb4lbZvuXDPGUev3hnA7 (ttl 60s)
  auditoría sys/leases/revoke: 1
  log operacional: 1

Lease que se deja caducar:
  creado database/creds/readonly/Zu3jj8mTpA0Iar4zUds9QzN6 (ttl 60s). Esperando 80s.
  log operacional al expirar: 1
  revocaciones manuales de este lease en auditoría: 0


## Comprobación final

En Grafana abre **Dashboards → Vault SOC** y **Alerting → Alert rules**. La evaluación es cada diez segundos.
No hay destinos externos de notificación ni SMTP.

La celda muestra el valor de cada regla en la ventana móvil. El estado en Grafana puede ir unos segundos por detrás.

In [49]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
echo "Valores actuales de las reglas (ventana móvil):"
jq -c '.groups[0].rules[]' "${WORKDIR}/rules.json" > "${WORKDIR}/rules.ndjson"
while IFS= read -r rule; do
  title=$(printf '%s' "$rule" | jq -r .title)
  threshold=$(printf '%s' "$rule" | jq -r '.data[1].model.conditions[0].evaluator.params[0]')
  expr=$(printf '%s' "$rule" | jq -r '.data[0].model.expr')
  value=$(loki_instant "$expr")
  state=$(awk -v v="$value" -v t="$threshold" 'BEGIN { if (v+0 > t+0) print "cumplida"; else print "no cumplida" }')
  echo "${title}: ${value} · condición > ${threshold} · ${state}"
done < "${WORKDIR}/rules.ndjson"
echo "Reglas: http://127.0.0.1:${GRAFANA_PORT}/alerting/list"

Valores actuales de las reglas (ventana móvil):
Fallos de login en la demo: 6 · condición > 4 · cumplida
Accesos denegados en la demo: 7 · condición > 4 · cumplida
Ráfaga de lecturas en la demo: 33 · condición > 19 · cumplida
Borrado o destrucción de secretos de demo: 3 · condición > 0 · cumplida
Uso de la política root en Vault: 44 · condición > 0 · cumplida
Cambios de políticas de demo: 4 · condición > 0 · cumplida
Cambios en dispositivos de auditoría: 2 · condición > 0 · cumplida
Revocación de leases por prefijo creds/readonly: 4 · condición > 0 · cumplida
Sin heartbeat de stdout durante 2 minutos: 0 · condición > 0 · no cumplida
Reglas: http://127.0.0.1:13000/alerting/list


## Correlación por request.id

Una lectura del lector y las dos líneas de auditoría (petición y respuesta) con el mismo `request.id`.
No se imprime el secreto. Si el token guardado ya no vale, se entra de nuevo como lector.

In [50]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
if ! VAULT_TOKEN="$LECTOR_TOKEN" vault token lookup >/dev/null 2>&1; then
  login_user lector "$LECTOR_PASSWORD"
fi
START=$(now_ns)
VAULT_TOKEN="$LECTOR_TOKEN" vault kv get "${DEMO_ID}/allowed/secret-00" >/dev/null
base=$(jq -r .reads "${WORKDIR}/queries.json")
wait_loki "${base} | path=\"${DEMO_ID}/data/allowed/secret-00\"" 1 "$START" >/dev/null
line=$(loki_last_line "${base} | path=\"${DEMO_ID}/data/allowed/secret-00\"" "$START")
req=$(printf '%s' "$line" | jq -r '.request.id')
test -n "$req"
audit=$(jq -r .audit "${WORKDIR}/queries.json")
echo "Consulta para pegar en Explore:"
echo "${audit} | request_id=\"${req}\""
curl -sfG "http://127.0.0.1:${LOKI_PORT}/loki/api/v1/query_range" \
  --data-urlencode "query=${audit} | request_id=\"${req}\"" \
  --data-urlencode "start=${START}" \
  --data-urlencode "end=$(date +%s)000000000" \
  --data-urlencode "limit=20" \
  --data-urlencode "direction=forward" \
  | jq -r '[.data.result[].values[][1] | fromjson | .type] | unique | .[]' > "${WORKDIR}/corr-types.txt"
grep -qx request "${WORKDIR}/corr-types.txt"
grep -qx response "${WORKDIR}/corr-types.txt"
rm -f "${WORKDIR}/corr-types.txt"
echo "Petición y respuesta correlacionadas. request.id=${req}"

Consulta para pegar en Explore:
{namespace="vault", log_kind="vault_audit"} | json event_type="type", path="request.path", operation="request.operation", error="error", principal="auth.display_name", policies="auth.policies", request_id="request.id", origin="request.remote_address" | __error__="" | request_id="efe97d63-7dd8-28b5-f529-2d44b4c4c9d8"
Petición y respuesta correlacionadas. request.id=efe97d63-7dd8-28b5-f529-2d44b4c4c9d8


## Prueba opcional: ausencia de logs

Para solo el heartbeat, deja Alloy recogiendo y espera 150 segundos. Cambia `PROBAR_AUSENCIA` a `true` para ejecutarla.
Si interrumpes la celda, el `trap` vuelve a levantar el heartbeat. Si cierras el kernel a medias, aplica otra vez el Deployment.

In [51]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
PROBAR_AUSENCIA=false
if [ "$PROBAR_AUSENCIA" != true ]; then
  echo "Prueba opcional omitida."
  exit 0
fi
kube scale deployment/soc-heartbeat -n "$VAULT_K8S_NAMESPACE" --replicas=0
trap 'kube scale deployment/soc-heartbeat -n "$VAULT_K8S_NAMESPACE" --replicas=1' EXIT
echo "Heartbeat parado. Sigue su alerta en Grafana durante 150 segundos."
sleep 150
expr=$(jq -r '.groups[0].rules[] | select(.uid=="vault-soc-pipeline") | .data[0].model.expr' "${WORKDIR}/rules.json")
value=$(loki_instant "$expr")
awk -v v="$value" 'BEGIN { exit !(v+0 > 0) }'
kube scale deployment/soc-heartbeat -n "$VAULT_K8S_NAMESPACE" --replicas=1
trap - EXIT
wait_loki "$(jq -r .heartbeat "${WORKDIR}/queries.json")" 1 "$(now_ns)" >/dev/null
echo "Heartbeat restaurado y recibido en Loki."

Prueba opcional omitida.


## Diagnóstico

Si no llegan datos, revisa en orden: pods y PVC, Alloy, Loki y el JSON de auditoría en la salida de Vault.
Si reinicia el pod del port-forward, cierra los forwards y vuelve a abrirlos.
Si caducan los tokens, la celda de correlación entra de nuevo; si falta `demo.env`, vuelve a crear el laboratorio.

In [52]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
kube get pods,pvc,svc -n "$OBS_NAMESPACE"
for name in alloy loki grafana; do
  echo ""
  echo "--- ${name}: últimas líneas ---"
  kube logs -n "$OBS_NAMESPACE" "deployment/${name}" --tail=15 || true
done
echo "Consulta de todos los logs de Vault: {namespace=\"${VAULT_K8S_NAMESPACE}\"}"

NAME                           READY   STATUS    RESTARTS   AGE
pod/alloy-5cc6f7f758-2dn7c     1/1     Running   0          2m22s
pod/grafana-755dbfd568-j77f2   1/1     Running   0          2m22s
pod/loki-7bb7f79d88-tx99k      1/1     Running   0          2m22s

NAME                                 STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS   VOLUMEATTRIBUTESCLASS   AGE
persistentvolumeclaim/grafana-data   Bound    pvc-20f34051-9ab4-45f2-a7f6-6fdd514e6508   1Gi        RWO            standard       <unset>                 5h8m
persistentvolumeclaim/loki-data      Bound    pvc-aa12a4bf-4926-4afa-9ab2-3947d0076f4e   5Gi        RWO            standard       <unset>                 5h8m

NAME              TYPE        CLUSTER-IP       EXTERNAL-IP   PORT(S)    AGE
service/grafana   ClusterIP   10.110.234.155   <none>        3000/TCP   5h8m
service/loki      ClusterIP   10.108.70.101    <none>        3100/TCP   5h8m

--- alloy: últimas líneas ---

## Limpieza opcional

Las tres variables empiezan en `false` para que ejecutar la celda no borre la demo.
Pon en `true` solo lo que quieras retirar y vuelve a ejecutarla.

- `BORRAR_DATOS_DEMO`: revoca los tokens y elimina el KV, el userpass y las políticas de `DEMO_ID`.
- `RETIRAR_AUDITORIA_DEMO`: retira `soc-stdout` y `soc-file` solo si queda otro dispositivo de auditoría. No borra los ficheros históricos.
- `BORRAR_OBSERVABILIDAD`: elimina los recursos de esta demo, incluidos los PVC. Conserva el namespace `vault-observability`.

No se elimina Vault ni su namespace.

In [53]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
BORRAR_DATOS_DEMO=false
RETIRAR_AUDITORIA_DEMO=false
BORRAR_OBSERVABILIDAD=false

delete_owned() {
  local kind="$1" name="$2" ns="$3" owner
  if ! owner=$(kube get "$kind" "$name" -n "$ns" -o jsonpath='{.metadata.labels.app\.kubernetes\.io/part-of}' 2>/dev/null); then
    return 0
  fi
  if [ "$owner" != "$OWNER" ]; then
    echo "No se borra un recurso ajeno: ${kind}/${name}" >&2
    exit 1
  fi
  kube delete "$kind" "$name" -n "$ns" --wait=false
}

if [ "$BORRAR_DATOS_DEMO" = true ]; then
  if [ -n "${LECTOR_TOKEN:-}" ]; then
    VAULT_TOKEN="$LECTOR_TOKEN" vault token revoke -self >/dev/null 2>&1 || true
  fi
  if [ -n "${OPERADOR_TOKEN:-}" ]; then
    VAULT_TOKEN="$OPERADOR_TOKEN" vault token revoke -self >/dev/null 2>&1 || true
  fi
  vault secrets disable "$DEMO_ID" >/dev/null 2>&1 || true
  vault auth disable "$DEMO_ID" >/dev/null 2>&1 || true
  vault policy delete "${DEMO_ID}-reader" >/dev/null 2>&1 || true
  vault policy delete "${DEMO_ID}-operator" >/dev/null 2>&1 || true
  rm -f "${WORKDIR}/demo.env"
  echo "Recursos de prueba de Vault eliminados."
fi

if [ "$RETIRAR_AUDITORIA_DEMO" = true ]; then
  others=$(vault audit list -format=json | jq '[keys[] | select(. != "soc-stdout/" and . != "soc-file/")] | length')
  if [ "$others" -eq 0 ]; then
    echo "Se conserva la auditoría: no existe otro destino fuera de esta demo."
  else
    vault audit disable soc-stdout >/dev/null 2>&1 || true
    vault audit disable soc-file >/dev/null 2>&1 || true
    echo "Retirados los dispositivos de esta demo."
  fi
fi

if [ "$BORRAR_OBSERVABILIDAD" = true ]; then
  delete_owned deployment loki "$OBS_NAMESPACE"
  delete_owned deployment alloy "$OBS_NAMESPACE"
  delete_owned deployment grafana "$OBS_NAMESPACE"
  delete_owned deployment soc-heartbeat "$VAULT_K8S_NAMESPACE"
  delete_owned service loki "$OBS_NAMESPACE"
  delete_owned service grafana "$OBS_NAMESPACE"
  delete_owned configmap loki-config "$OBS_NAMESPACE"
  delete_owned configmap alloy-config "$OBS_NAMESPACE"
  delete_owned configmap grafana-datasources "$OBS_NAMESPACE"
  delete_owned configmap grafana-providers "$OBS_NAMESPACE"
  delete_owned configmap grafana-dashboards "$OBS_NAMESPACE"
  delete_owned configmap grafana-alerting "$OBS_NAMESPACE"
  delete_owned persistentvolumeclaim loki-data "$OBS_NAMESPACE"
  delete_owned persistentvolumeclaim grafana-data "$OBS_NAMESPACE"
  delete_owned secret grafana-admin "$OBS_NAMESPACE"
  delete_owned serviceaccount alloy "$OBS_NAMESPACE"
  delete_owned role vault-soc-logs "$VAULT_K8S_NAMESPACE"
  delete_owned rolebinding vault-soc-logs "$VAULT_K8S_NAMESPACE"
  echo "Eliminados los recursos de observabilidad de esta demo. El namespace se conserva."
fi

if [ "$BORRAR_DATOS_DEMO" != true ] && [ "$RETIRAR_AUDITORIA_DEMO" != true ] && [ "$BORRAR_OBSERVABILIDAD" != true ]; then
  echo "Nada que borrar. Cambia las variables a true para limpiar."
fi

Nada que borrar. Cambia las variables a true para limpiar.


### Cerrar los accesos locales

Cuando termines con Grafana, pon `CERRAR_ACCESOS=true` y ejecuta la celda.
Solo cierra los port-forward cuyo PID guardó este cuaderno.

In [54]:
%%bash
source "${WORKDIR:-/tmp/vault/soc}/lab.sh"
load_env
CERRAR_ACCESOS=false
if [ "$CERRAR_ACCESOS" != true ]; then
  echo "Cambia CERRAR_ACCESOS=true para cerrar los port-forward."
  exit 0
fi
for f in "${WORKDIR}"/pf-*.pid; do
  [ -e "$f" ] || continue
  kill "$(cat "$f")" 2>/dev/null || true
  rm -f "$f"
done
echo "Port-forwards de esta sesión cerrados."

Cambia CERRAR_ACCESOS=true para cerrar los port-forward.


## Referencias

- [Vault: auditoría en fichero/stdout](https://developer.hashicorp.com/vault/docs/audit/file)
- [Vault: esquema y correlación de peticiones/respuestas](https://developer.hashicorp.com/vault/docs/audit/schema)
- [Vault: buenas prácticas y varios destinos de auditoría](https://developer.hashicorp.com/vault/docs/audit/best-practices)
- [Vault: ejemplos de respuesta a incidentes](https://developer.hashicorp.com/vault/tutorials/monitoring/audit-elastic-incident-response#incident-response-examples)
- [Vault: API de KV v2](https://developer.hashicorp.com/vault/api-docs/secret/kv/kv-v2)
- [Vault: API de políticas](https://developer.hashicorp.com/vault/api-docs/system/policies)
- [Vault: bloqueo por intentos fallidos](https://developer.hashicorp.com/vault/api-docs/system/auth#tune-auth-method)
- [Alloy: logs por la API de Kubernetes](https://grafana.com/docs/alloy/latest/reference/components/loki/loki.source.kubernetes/)
- [Alloy: procesamiento de logs](https://grafana.com/docs/alloy/latest/reference/components/loki/loki.process/)
- [Loki: configuración](https://grafana.com/docs/loki/latest/configure/examples/)
- [Loki: consultas LogQL](https://grafana.com/docs/loki/latest/query/log_queries/)
- [Grafana: aprovisionamiento](https://grafana.com/docs/grafana/latest/administration/provisioning/)
- [Grafana: aprovisionamiento de alertas](https://grafana.com/docs/grafana/latest/alerting/set-up/provision-alerting-resources/file-provisioning/)